# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

## Step 1: Install required packages

In [1]:
!pip install -U langchain python-dotenv langchain-openai langchain-community langchain-classic langchain-text-splitters faiss-cpu tiktoken pypdf

## Step 2: Import dependencies

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import (
    AzureChatOpenAI,
    AzureOpenAIEmbeddings,
    ChatOpenAI,
    OpenAIEmbeddings,
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

/var/folders/cl/9xdnhz2j0c730whpw8n1s9cw0000gn/T/ipykernel_58227/2294427799.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 3: Set credentials
Loaded from the shared `.env` at the repo root. Uses OpenAI if `OPENAI_API_KEY` is set, otherwise falls back to Azure OpenAI.

In [3]:
load_dotenv(find_dotenv())

if os.environ.get("OPENAI_API_KEY"):
    provider = "openai"
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
elif os.environ.get("AZURE_OPENAI_API_KEY"):
    provider = "azure"
    llm = AzureChatOpenAI(
        azure_endpoint="https://openai-api-management-gw.azure-api.net",
        api_version="2025-01-01-preview",
        deployment_name="gpt-5-mini",
    )
    embeddings = AzureOpenAIEmbeddings(
        azure_endpoint="https://openai-api-management-gw.azure-api.net",
        api_version="2023-05-15",
        deployment="text-embedding-ada-002",
    )
else:
    raise RuntimeError("Set OPENAI_API_KEY or AZURE_OPENAI_API_KEY in the .env file")

print(f"Using provider: {provider}")

Using provider: openai


### Quick connectivity test
A plain prompt, no FAQ document needed.

In [4]:
test_response = llm.invoke("Say 'connection ok' if you can read this.")
print("Test prompt response:", test_response.content)

Test prompt response: Connection ok.


## Step 4: Choose a menu

In [13]:
available_cuisines = ["sushi", "steak", "italian"]
cuisine = input(f"Which menu would you like ({'/'.join(available_cuisines)})? ").strip().lower()
while cuisine not in available_cuisines:
    cuisine = input(f"Please choose one of {'/'.join(available_cuisines)}: ").strip().lower()
menu = f"menus/{cuisine}.pdf"

### 2.1 Load and chunk PDFs using PyPDFLoader

In [14]:
loader = PyPDFLoader(menu)
documents = loader.load()

### 2.2 Load and chunk PDFs using RecursiveCharacterTextSplitter

In [15]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

### 2.3 Generate embeddings and store in FAISS vectorstore

In [16]:
vectorstore = FAISS.from_documents(docs, embeddings)

### 2.4 Run retrieval queries with GPT to fetch contextually relevant chunks

In [17]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True,
)

## Step 5: Analyze the chunking results
Total chunk count and average chunk size.

In [18]:
chunk_sizes = [len(doc.page_content) for doc in docs]
print(f"\nTotal chunks: {len(docs)}")
print(f"Average chunk size: {sum(chunk_sizes) / len(chunk_sizes):.0f} characters")


Total chunks: 31
Average chunk size: 376 characters


## Step 6: Sample query and top matching result

In [19]:
query = "What sushi rolls are on the menu?"
result = qa_chain.invoke({"query": query})

print("\nAnswer:", result["result"])
print("\n--- Top matching result ---")
print(result["source_documents"][0].page_content)


Answer: The sushi rolls on the menu are:

1. BBQ Freshwater Eel Roll - $17.50
2. Spicy Tuna Volcano Roll - $17.50
3. Rainbow Roll - $19.00
4. Salmon Lover Roll - $18.00
5. Crispy Tiger Roll - $16.50
6. Truffle Yellowtail Roll - $20.00
7. Spicy Scallop Hand Roll (2 pcs) - $15.00
8. A5 Miyazaki Wagyu Roll - $28.00
9. Dragon Roll - $18.50

--- Top matching result ---
-  BBQ  freshwater  eel  and  crisp  cucumber  inside,  topped  with  thinly  sliced  avocado,  sweet  unagi  glaze,  and  toasted  sesame  seeds.  ●  Spicy  Tuna  Volcano  Roll  ($17.50)  [Pescatarian  |  Spicy]  -  Spicy  minced  tuna  and  cucumber,  topped  with  baked  spicy  crab  salad,  crispy  tempura  ﬂakes,  and  sriracha  unagi  glaze.  ●  Rainbow  Roll  ($19.00)  [Pescatarian  |  Mild]  -  Crab  salad  and  cucumber  roll  draped  with  fresh  slices  of  raw  Atlantic  salmon,


## Step 7: Ask your own question
Loops until you type `q` to quit.

In [20]:
while True:
    user_query = input(f"\nWhat would you like to know about the {cuisine} menu? (q to quit) ")
    if user_query.strip().lower() == "q":
        break
    user_result = qa_chain.invoke({"query": user_query})
    print("\nAnswer:", user_result["result"])


Answer: The Rainbow Roll is a sushi roll that consists of crab salad and cucumber, draped with fresh slices of raw Atlantic salmon and topped with a sweet tare glaze. It costs $19.00.

Answer: I don't know.

Answer: I don't know.
